In [1]:
import pandas as pd

In [2]:
train_set = pd.read_csv("data/train_normal_rtf.csv")
test_set = pd.read_csv("data/test_normal_rtf.csv")

In [3]:
print("Train class distribution:")
print(train_set['label'].value_counts())
print(train_set['label'].value_counts(normalize=True).rename('proportion'))

print("\nTest class distribution:")
print(test_set['label'].value_counts())
print(test_set['label'].value_counts(normalize=True).rename('proportion'))

Train class distribution:
label
1    182
0    174
Name: count, dtype: int64
label
1    0.511236
0    0.488764
Name: proportion, dtype: float64

Test class distribution:
label
1    20
0    20
Name: count, dtype: int64
label
1    0.5
0    0.5
Name: proportion, dtype: float64


In [ ]:
feature_cols = [str(i) for i in range(70)]

train_stats = train_set[feature_cols].agg(['mean', 'std']).T
test_stats  = test_set[feature_cols].agg(['mean', 'std']).T

comparison = train_stats.rename(columns={'mean': 'Train Mean', 'std': 'Train SD'})
comparison[['Test Mean', 'Test SD']] = test_stats[['mean', 'std']].values
print(comparison)

In [ ]:
from scipy import stats

results = []
for col in feature_cols:
    stat, pval = stats.mannwhitneyu(train_set[col], test_set[col], alternative='two-sided')
    results.append({'Feature': col, 'MWU_stat': stat, 'p_value': pval})

results_df = pd.DataFrame(results)

# Apply Bonferroni correction for 70 features
alpha_corrected = 0.05 / 70
sig_count = (results_df['p_value'] < alpha_corrected).sum()
print(f"Features with significant difference (Bonferroni-corrected): {sig_count} / 70")

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

X_train = train_set[feature_cols].values
X_test  = test_set[feature_cols].values

pca = PCA(n_components=2)
pca.fit(X_train)

train_pca = pca.transform(X_train)
test_pca  = pca.transform(X_test)

plt.figure(figsize=(7, 5))
plt.scatter(train_pca[:, 0], train_pca[:, 1], alpha=0.5, label='Train', s=20)
plt.scatter(test_pca[:, 0],  test_pca[:, 1],  alpha=0.5, label='Test',  s=20, marker='^')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA: Overlap of Train and Test Sets')
plt.legend()
plt.tight_layout()
plt.savefig('pca_train_test_overlap.pdf')
plt.show()

In [ ]:
for label_val, label_name in [(1, 'Positive'), (0, 'Negative')]:
    n_train = (train_set['label'] == label_val).sum()
    n_test  = (test_set['label']  == label_val).sum()
    total   = n_train + n_test
    print(f"{label_name}: Train={n_train} ({n_train/total:.1%}), Test={n_test} ({n_test/total:.1%})")

## Class distribution: training vs test (Table 1)

Pearson chi-square test of independence on the 2×2 table (partition × label), **without** Yates' correction:
this is the method behind p = 0.894 in Table 1 of the submitted manuscript (recomputed here: 0.893).

The old value (40-row test set, which held 10 SMOTE rows) is compared with the new 30-patient test set, using
all training rows and the real training patients only (the 91 SMOTE training rows left out).

In [4]:
from scipy.stats import chi2_contingency

# p-value printed in Table 1 of the submitted manuscript (old test set)
SUBMITTED_P_VALUE = 0.894

# rtfDataSet.csv rows 0-294 are the 295 real patients; rows 295-395 are the 101 SMOTE rows (all label 1).
# data/split_raw_ids.csv gives, for each row of train_normal_rtf.csv (split_row), its row in rtfDataSet.csv (raw_id).
N_REAL_PATIENTS = 295
train_ids = pd.read_csv('data/split_raw_ids.csv').query("split == 'train'").sort_values('split_row')
assert (train_ids['label'].to_numpy() == train_set['label'].to_numpy()).all(), 'split_raw_ids.csv does not match the training file'
real_train_labels = train_ids.loc[train_ids['raw_id'] < N_REAL_PATIENTS, 'label']

CLASS_TEST_SETS = {
    'old (submitted, 40 rows)': pd.read_csv('data/test_normal_rtf.csv')['label'],     # 30 real patients + 10 SMOTE rows
    'new (30 real patients)': pd.read_csv('data/test_rtf_v2_cleaned.csv')['label'],
}
CLASS_TRAIN_SETS = {
    'all training rows': train_set['label'],            # 265 real patients + 91 SMOTE rows
    'real training patients': real_train_labels,        # SMOTE rows left out
}
print({name: len(labels) for name, labels in {**CLASS_TRAIN_SETS, **CLASS_TEST_SETS}.items()})

{'all training rows': 356, 'real training patients': 265, 'old (submitted, 40 rows)': 40, 'new (30 real patients)': 30}


In [5]:
def label_counts(labels):
    """(label 1, label 0) counts."""
    return int((labels == 1).sum()), int((labels == 0).sum())


def with_share(n, total):
    return f'{n} ({n / total:.1%})'


rows = []
for test_name, test_labels in CLASS_TEST_SETS.items():
    for train_name, train_labels in CLASS_TRAIN_SETS.items():
        (tr1, tr0), (te1, te0) = label_counts(train_labels), label_counts(test_labels)
        chi2, p, _, _ = chi2_contingency([[tr1, tr0], [te1, te0]], correction=False)
        rows.append({
            'test set': test_name,
            'training': train_name,
            'train N': tr1 + tr0, 'train label 1': with_share(tr1, tr1 + tr0), 'train label 0': with_share(tr0, tr1 + tr0),
            'test N': te1 + te0, 'test label 1': with_share(te1, te1 + te0), 'test label 0': with_share(te0, te1 + te0),
            'chi2': round(chi2, 3),
            'p-value': round(p, 3),
        })
class_distribution = pd.DataFrame(rows)

old = class_distribution.iloc[0]
print(f'Submitted Table 1: p = {SUBMITTED_P_VALUE} | recomputed ({old["test set"]}, {old["training"]}): '
      f'p = {old["p-value"]:.3f}')
class_distribution

Submitted Table 1: p = 0.894 | recomputed (old (submitted, 40 rows), all training rows): p = 0.893


,test set,training,train N,train label 1,train label 0,test N,test label 1,test label 0,chi2,p-value
0,"old (submitted, 40 rows)",all training rows,356,182 (51.1%),174 (48.9%),40,20 (50.0%),20 (50.0%),0.018,0.893
1,"old (submitted, 40 rows)",real training patients,265,91 (34.3%),174 (65.7%),40,20 (50.0%),20 (50.0%),3.682,0.055
2,new (30 real patients),all training rows,356,182 (51.1%),174 (48.9%),30,10 (33.3%),20 (66.7%),3.503,0.061
3,new (30 real patients),real training patients,265,91 (34.3%),174 (65.7%),30,10 (33.3%),20 (66.7%),0.012,0.912
